# Module 2.2 - Build Gate Starter: Your Own Autograd Engine

Fill in every `TODO`. Do not import torch except where the gradient check uses it.
When the gradient check passes and the MLP learns, the build gate is green.

Companion lesson: `02-autograd-from-numpy.md`. Build it yourself first; only peek at
Karpathy's micrograd repo when truly stuck.

## 1. The `Value` class

A scalar wrapped with the bookkeeping needed for reverse-mode autodiff: its data, its
gradient, the children that produced it, and a `_backward` closure that pushes gradient
to those children using the local derivative.

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None   # set by each operation
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # TODO: gradient of a sum flows through unchanged to both children.
            # remember to ACCUMULATE (+=), not overwrite.
            pass
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # TODO: local grads are other.data and self.data. accumulate.
            pass
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), 'only int/float powers'
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            # TODO: d/dx (x**n) = n * x**(n-1). accumulate.
            pass
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            # TODO: d/dx tanh(x) = 1 - tanh(x)**2 = 1 - t**2. accumulate.
            pass
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            # TODO: grad passes through only where data > 0. accumulate.
            pass
        out._backward = _backward
        return out

    def backward(self):
        # build reverse topological order, seed grad=1.0, walk backward
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    # convenience operators (these should just work once the above are correct)
    def __neg__(self):        return self * -1
    def __radd__(self, o):    return self + o
    def __sub__(self, o):     return self + (-o)
    def __rsub__(self, o):    return (-self) + o
    def __rmul__(self, o):    return self * o
    def __truediv__(self, o): return self * (o ** -1 if isinstance(o,(int,float)) else o**-1)
    def __repr__(self):       return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'

## 2. Gradient check vs PyTorch (the correctness test)

Build the same expression in your engine and in torch, call backward in both, and assert
the gradients match. This is the non-negotiable test. It should pass once your `_backward`
closures are correct.

In [ ]:
import torch

def test_against_torch():
    # your engine
    a = Value(-4.0); b = Value(2.0)
    c = a + b
    d = a * b + b**3
    e = (c + d).tanh()
    f = e * d
    f.backward()

    # torch reference
    at = torch.tensor(-4.0, requires_grad=True)
    bt = torch.tensor(2.0,  requires_grad=True)
    ct = at + bt
    dt = at * bt + bt**3
    et = torch.tanh(ct + dt)
    ft = et * dt
    ft.backward()

    assert abs(f.data - ft.item()) < 1e-6, 'forward mismatch'
    assert abs(a.grad - at.grad.item()) < 1e-6, f'da: {a.grad} vs {at.grad.item()}'
    assert abs(b.grad - bt.grad.item()) < 1e-6, f'db: {b.grad} vs {bt.grad.item()}'
    print('PASS: gradients match PyTorch.')

test_against_torch()

## 3. A tiny neural net on top of `Value`

Build `Neuron`, `Layer`, `MLP` using only your `Value`. No numpy in the forward pass
(scalars only), no torch.

In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        # TODO: act = sum(wi*xi) + b, then return act.tanh()
        pass
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## 4. Train it (the 'it learns' test)

Fit `make_moons` with your own SGD loop. Watch the loss fall. Remember to ZERO the grads
each step before calling backward.

In [ ]:
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=100, noise=0.1, random_state=0)
y = [1.0 if yi == 1 else -1.0 for yi in y]   # map to {-1, +1} for tanh output

model = MLP(2, [16, 16, 1])

def loss_fn():
    # TODO: forward all points, compute mean squared error (pred - target)**2,
    # return the total/mean loss Value.
    pass

losses = []
for step in range(100):
    loss = loss_fn()
    for p in model.parameters():   # zero grads
        p.grad = 0.0
    loss.backward()
    lr = 0.05
    for p in model.parameters():   # SGD step
        p.data -= lr * p.grad
    losses.append(loss.data)
    if step % 10 == 0:
        print(f'step {step:3d}  loss {loss.data:.4f}')

plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss'); plt.title('Your engine learning'); plt.show()

## Done?

- Gradient check prints PASS
- Loss curve decreases to a good fit
- You can derive every `_backward` on paper

Then write the one-page derivation report and package the engine. That is your Capstone 0
seed. See the ship gate in the lesson file.